# Safety-Incident Anomaly Pattern Detection

**INT396 · SIH26165**  
This notebook is part of a reproducible project comparing Isolation Forest and Local Outlier Factor on OSHA Severe Injury Reports.

**Interpretation rule:** model flags are statistical outliers, not verified anomalies, accident predictions, or proof of unsafe conduct.

Select the project's Python environment as the notebook kernel. The raw dataset must be available at `data/raw/January2015toNovember2025.csv`.

# Feature Engineering and Preprocessing

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
from safety_anomaly.data.io import load_incidents
from safety_anomaly.data.clean import prepare_records
from safety_anomaly.features.build import build_features
DATA_PATH=ROOT/'data/raw/January2015toNovember2025.csv'
df=prepare_records(load_incidents(DATA_PATH))
SAMPLE_SIZE=min(5000,len(df))
sample=df.sample(n=SAMPLE_SIZE, random_state=42).sort_index().reset_index(drop=True)
print('Modeling sample:', sample.shape)

## 1. Feature governance

In [ ]:
excluded=['ID','UPA','Employer','Address1','Address2','City','State','Zip','Latitude','Longitude','Inspection','Hospitalized','Amputation','Loss of Eye','FederalState']
base=['Primary NAICS','EventTitle','SourceTitle','Secondary Source Title']
text=['Final Narrative']
supplementary=['NatureTitle','Part of Body Title']
print('Excluded from primary features:', excluded)
print('Base structured fields:', base)
print('Text field:', text)
print('Supplementary fields:', supplementary)

## 2. Compare feature representations

In [ ]:
feature_sets={}
for name in ['structured','structured_injury','structured_text']:
    result=build_features(sample, representation=name, random_state=42,
                          svd_components=24, min_category_frequency=15,
                          max_text_features=2500, text_min_df=3)
    feature_sets[name]=result
    print(name, 'matrix shape=', result.X.shape,
          'finite=', bool(np.isfinite(result.X).all()))

## 3. Notes on dimensionality reduction
One-hot and TF-IDF representations can be sparse and high-dimensional. TruncatedSVD reduces this combined sparse representation to a compact dense vector. StandardScaler is then applied to the reduced components. This is a computational representation choice, not evidence that the latent components have direct semantic meaning.

## 4. Leakage and privacy checks

In [ ]:
assert not set(excluded).intersection(base + text)
assert 'Hospitalized' not in base and 'Amputation' not in base and 'Loss of Eye' not in base
print('Feature governance assertions passed.')